In [5]:
import os, torch, random
import torchvision
import torchvision.transforms as transforms
from torch import nn
from torch.utils.data import DataLoader
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

BASE_DIR = r"D:\pycharm\CNN"
DATA_DIR = os.path.join(BASE_DIR, "data", "food-11")
MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [6]:
train_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                     [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                     [0.229, 0.224, 0.225])
])

train_ds = torchvision.datasets.ImageFolder(os.path.join(DATA_DIR, "training", "labeled"), transform=train_tf)
val_ds = torchvision.datasets.ImageFolder(os.path.join(DATA_DIR, "validation"), transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

In [7]:
class BetterCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 11)
        )

    def forward(self, x): return self.net(x)


model = BetterCNN().to(device)

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2)

best = 0
wait = 0


def evaluate():
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100 * correct / total


for epoch in range(30):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = criterion(model(x), y)
        optimizer.zero_grad();
        loss.backward();
        optimizer.step()

    acc = evaluate()
    scheduler.step(acc)
    print(f"Epoch {epoch + 1}: {acc:.2f}%")

    if acc > best:
        best = acc
        wait = 0
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, "level2.pth"))
    else:
        wait += 1
        if wait >= 5:
            print("Early stop")
            break

print("Best:", best)

Epoch 1: 18.33%
Epoch 2: 19.85%
Epoch 3: 28.03%
Epoch 4: 28.64%
Epoch 5: 31.06%
Epoch 6: 33.03%
Epoch 7: 33.18%
Epoch 8: 35.30%
Epoch 9: 29.24%
Epoch 10: 35.30%
Epoch 11: 33.48%
Epoch 12: 36.06%
Epoch 13: 37.42%
Epoch 14: 38.18%
Epoch 15: 38.18%
Epoch 16: 38.64%
Epoch 17: 37.42%
Epoch 18: 38.03%
Epoch 19: 37.42%
Epoch 20: 38.03%
Epoch 21: 37.88%
Early stop
Best: 38.63636363636363
